In [44]:
# Imports
import numpy as np
import pandas as pd
import tensorflow as tf
from collections import Counter
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM, Dropout, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from prepare_data import clean_sentences, combine_reviews

In [45]:
# Load the CSV file
csv_data = 'data/data_reduced.csv' # Changed for each machine.
data = pd.read_csv(csv_data)

In [46]:
# Initialize the lists
positive_reviews = []
negative_reviews = []
neutral_reviews = []

# Clean the reviews and split them into positive, negative, and neutral
def review_splitting(row):
    cleaned_review = clean_sentences(pd.Series([row['1']]))[0]
    if row['0'] >= 8:
        positive_reviews.append(cleaned_review)
    elif row['0'] <= 4:
        negative_reviews.append(cleaned_review)
    else:
        neutral_reviews.append(cleaned_review)

# Applies review_splitting() to each row in the dataset
result = data.apply(review_splitting, axis = 1)

# Labels the reviews and convert to a numpy array
labels = [0] * len(negative_reviews) + [1] * len(neutral_reviews) + [2] * len(positive_reviews)
labels = np.array(labels)

combined_list = negative_reviews + neutral_reviews + positive_reviews

In [47]:
# Check the total number of words after sentence cleaning
total_words = []
for sentence in combined_list:
    words = sentence.split()
    for word in words:
        total_words.append(word)

# Check the number of unique words after sentence cleaning
unique_words = []
for sentence in combined_list:
    words = sentence.split()
    for word in words:
        if word not in unique_words:
            unique_words.append(word)

# Find the 95th percentile of the sequence lengths to not guess maxlen in the pad sequences
sequence_lengths = []
for review in combined_list:
    words = review.split()
    length = len(words)
    sequence_lengths.append(length)
percentile_95 = np.percentile(sequence_lengths, 95)

# Check how many words that appear more than 5 times. This is to better determine the size of num_words in the Tokenizer
data_count = Counter(total_words)
word_frequency = []
for word, count in data_count.items():
    if count >= 5:
        word_frequency.append(word)


print(f'Total words: {len(total_words)}')
print(f'Unique words: {len(unique_words)}')
print(f"95th Percentile Length: {percentile_95}")
print(f"Words that appear more than 5 times: {len(word_frequency)}")

Total words: 9707345
Unique words: 78936
95th Percentile Length: 53.0
Words that appear more than 5 times: 19787


In [48]:
# Tokenize the data
num_words = len(word_frequency)
tokenizer = Tokenizer(num_words = num_words)
tokenizer.fit_on_texts(combined_list)
sequences = tokenizer.texts_to_sequences(combined_list)

# Pad the sequences
maxlen = int(percentile_95)
pading = pad_sequences(sequences, maxlen = maxlen)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(pading, labels, test_size = 0.2, random_state = 42)

In [49]:
# Building the LSTM model
model = Sequential()
model.add(Embedding(input_dim = 20000, output_dim = 128, mask_zero = True))
model.add(LSTM(units = 128, return_sequences = False))
model.add(Dropout(0.5))
model.add(Dense(units = 3, activation = 'softmax'))

# LSTM - Vurder kernel_regularizer og recurrent_regularizer for å motvirke overfitting

In [50]:
# Compiling the model
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

# Training the model
model.fit(X_train, y_train, epochs = 10, batch_size = 1024, validation_data = (X_val, y_val), verbose = 1)

# Save the model
model.save('./data/model/sentiment_lstm_model.keras')

print('Model training complete and saved as "./data/model/sentiment_lstm_model.keras"')

Epoch 1/10
403/403 [==============================] - 591s 1s/step - loss: 0.5282 - accuracy: 0.7641 - val_loss: 0.4772 - val_accuracy: 0.7835
Epoch 2/10
403/403 [==============================] - 699s 2s/step - loss: 0.4652 - accuracy: 0.7917 - val_loss: 0.4718 - val_accuracy: 0.7869
Epoch 3/10
403/403 [==============================] - 765s 2s/step - loss: 0.4481 - accuracy: 0.8001 - val_loss: 0.4697 - val_accuracy: 0.7880
Epoch 4/10
403/403 [==============================] - 774s 2s/step - loss: 0.4342 - accuracy: 0.8067 - val_loss: 0.4723 - val_accuracy: 0.7873
Epoch 5/10
403/403 [==============================] - 755s 2s/step - loss: 0.4220 - accuracy: 0.8131 - val_loss: 0.4830 - val_accuracy: 0.7834
Epoch 6/10
403/403 [==============================] - 752s 2s/step - loss: 0.4091 - accuracy: 0.8195 - val_loss: 0.4904 - val_accuracy: 0.7843
Epoch 7/10
403/403 [==============================] - 740s 2s/step - loss: 0.3958 - accuracy: 0.8256 - val_loss: 0.5092 - val_accuracy: 0.7813

In [51]:
# Load the model
model = load_model('./data/model/sentiment_lstm_model.keras')

In [52]:
# Evaluate the model on the validation set
val_predictions = model.predict(X_val)
val_predictions = np.argmax(val_predictions, axis = 1)

# Print the classification report and confusion matrix
print(classification_report(y_val, val_predictions))
print(confusion_matrix(y_val, val_predictions))

3224/3224 [==============================] - 72s 22ms/step
              precision    recall  f1-score   support

           0       0.47      0.28      0.35      2168
           1       0.68      0.62      0.65     33844
           2       0.83      0.87      0.85     67136

    accuracy                           0.78    103148
   macro avg       0.66      0.59      0.62    103148
weighted avg       0.77      0.78      0.77    103148

[[  612  1373   183]
 [  599 21096 12149]
 [   83  8651 58402]]
